In [ ]:
# Data Collection Methods for Machine Learning — Demo (Expanded)
**Author:** Sayantan Samanta  
**Department of CSE:** ...  
**Date:** 25-08-2004

This notebook demonstrates a variety of ways to collect data for ML, following your handwritten list:
- Read from CSV / Excel
- Read from SQL database (SQLite example)
- REST API
- Web scraping (static) with BeautifulSoup
- Web scraping (dynamic) with Selenium (example guarded; needs webdriver)
- Streaming data (simulated Kafka-like stream)
- IoT / sensor data (simulated)
- Collect images from a local folder
- Audio data (generate & read simple WAV)
- Read from log files

All examples are self-contained and designed to run in a typical student environment. Sections that require external services (Selenium webdriver, real Kafka, serial device) are provided as guarded examples or simulated so you can demonstrate the concept without heavy infra.


In [ ]:
# Setup / imports used across cells
import os, io, sys, json, time, random
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import sqlite3
import pandas as pd
from PIL import Image
import numpy as np
import wave


## 1) Read data from CSV / Excel
Example: create a small DataFrame and read/write CSV and Excel (no external files required).

In [ ]:
# CSV / Excel demo using pandas (create in-memory dataframe then read it back)
df = pd.DataFrame({
    'student': ['Alice','Bob','Charlie'],
    'study_hours': [3.5, 2.0, 5.0],
    'attendance': [0.9, 0.6, 0.95]
})
print("Original DataFrame:")
display(df)

# write to CSV & Excel in /mnt/data for demonstration
csv_path = '/mnt/data/demo_students.csv'
excel_path = '/mnt/data/demo_students.xlsx'
df.to_csv(csv_path, index=False)
df.to_excel(excel_path, index=False)
print(f"Saved CSV -> {csv_path}")
print(f"Saved Excel -> {excel_path}")

# read back
df_csv = pd.read_csv(csv_path)
print('\nRead back CSV:')
display(df_csv)

## 2) Read data from SQL database (SQLite)
Create an SQLite DB, insert rows, and query it.

In [ ]:
# SQLite demo (file-based DB in /mnt/data)
db_path = '/mnt/data/demo_students.db'
conn = sqlite3.connect(db_path)
cur = conn.cursor()
cur.execute('CREATE TABLE IF NOT EXISTS students (name TEXT, study_hours REAL, attendance REAL)')
cur.execute('DELETE FROM students')  # clear for repeatability
cur.executemany('INSERT INTO students VALUES (?,?,?)', df.values.tolist())
conn.commit()

print(f"DB saved at {db_path}. Querying:")
for row in cur.execute('SELECT name, study_hours FROM students'):
    print(row)

conn.close()

## 3) Collect data via REST API
Using GitHub public API (no key needed).

In [ ]:
api_url = "https://api.github.com/users/octocat"
resp = requests.get(api_url)
print('Status code:', resp.status_code)
if resp.ok:
    data = resp.json()
    # show a few fields
    subset = {k: data.get(k) for k in ['login','id','public_repos','followers','created_at','html_url']}
    print('Selected fields from API:')
    print(json.dumps(subset, indent=2))
else:
    print('Failed to fetch API:', resp.text)

## 4) Web scraping from a static website (BeautifulSoup)
Scrape a sample page from 'Books to Scrape' (allowed for practice).

In [ ]:
url = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
r = requests.get(url)
if r.ok:
    soup = BeautifulSoup(r.text, 'html.parser')
    title = soup.find('h1').text.strip()
    price = soup.find('p', class_='price_color').text.strip()
    print('Title:', title)
    print('Price:', price)
else:
    print('Failed to fetch page:', r.status_code)

## 5) Web scraping from a dynamic website (Selenium)
This cell shows example code but is guarded. You need a webdriver (e.g., chromedriver) in PATH to actually run it. If you don't have it, this cell will print instructions instead.

In [ ]:
try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.options import Options

    chrome_opts = Options()
    chrome_opts.add_argument('--headless=new')
    # Attempt to start a webdriver (this will fail if driver is not installed)
    driver = webdriver.Chrome(options=chrome_opts)
    driver.get('https://httpbin.org/forms/post')
    h = driver.title
    print('Page title (via Selenium):', h)
    driver.quit()
except Exception as e:
    print('Selenium example could not run in this environment.')
    print('If you want to run it locally, install selenium and chromedriver, then run the example.')
    print('Example pip install: pip install selenium') 
    print('Error (brief):', e)

## 6) Streaming data demonstration (simulated Kafka-like stream)
This simulates a producer emitting messages and a consumer processing them. For a real Kafka setup you would use kafka-python or confluent-kafka.

In [ ]:
# Simulated streaming using Python generators/queues
import threading, queue, time

q = queue.Queue()

def producer(q, n=5, delay=0.5):
    for i in range(n):
        msg = {'timestamp': time.time(), 'value': random.random()}
        q.put(msg)
        print('Produced', msg)
        time.sleep(delay)
    q.put(None)  # sentinel

def consumer(q):
    while True:
        item = q.get()
        if item is None:
            print('Consumer received sentinel, exiting')
            break
        print('Consumed', item)

# Start threads to simulate streaming
t1 = threading.Thread(target=producer, args=(q,6,0.4))
t2 = threading.Thread(target=consumer, args=(q,))
t2.start(); t1.start()
t1.join(); t2.join()

print('Simulated streaming demo finished.')

## 7) IoT / Sensor data (simulated)
If you had a sensor or serial device, you'd read from it. Here we simulate sensor readings (temperature/humidity).

In [ ]:
# Simulate IoT sensor readings and show a small stream
def sensor_sim(n=6):
    for _ in range(n):
        yield {'time': time.time(), 'temp_c': round(20 + random.uniform(-2,3),2), 'humidity': round(40 + random.uniform(-5,5),1)}
        
print('Simulated sensor stream:')
for s in sensor_sim(6):
    print(s)
    time.sleep(0.3)

## 8) Collect images from a local folder
This cell creates two small sample images and shows how to load all images from a folder for ML preprocessing.

In [ ]:
img_dir = Path('/mnt/data/sample_images')
img_dir.mkdir(exist_ok=True)
# create two simple images (RGB)
for i in range(2):
    arr = (np.random.rand(64,64,3)*255).astype('uint8')
    im = Image.fromarray(arr)
    p = img_dir / f'image_{i+1}.png'
    im.save(p)
    print('Created', p)

# list and open images
images = list(img_dir.glob('*.png'))
print('\nFound images:')
for p in images:
    print(p)
    display(Image.open(p))

## 9) Audio data
Generate a short WAV (sine wave) and read its basic properties with the `wave` module.

In [ ]:
# generate a 1-second sine wave WAV file and then read it
sr = 44100
t = np.linspace(0, 1, int(sr), False)
freq = 440.0  # A4
sine = (0.5*np.sin(2*np.pi*freq*t) * (2**15-1)).astype(np.int16)
wav_path = '/mnt/data/demo_sine.wav'
with wave.open(wav_path, 'w') as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)  # 2 bytes = 16 bits
    wf.setframerate(sr)
    wf.writeframes(sine.tobytes())

print('Generated WAV at', wav_path)
# read properties
with wave.open(wav_path,'rb') as wf:
    print('Channels:', wf.getnchannels())
    print('Sample width (bytes):', wf.getsampwidth())
    print('Frame rate (Hz):', wf.getframerate())
    print('Number of frames:', wf.getnframes())

## 10) Read from a log file
Write a simple log file and demonstrate tailing/reading it.

In [ ]:
log_path = '/mnt/data/demo_app.log'
with open(log_path, 'w') as f:
    for i in range(10):
        f.write(f'{time.time()} - INFO - sample log line {i}\n')

print('Wrote log file at', log_path)
print('\nLast 5 lines (tail):')
with open(log_path) as f:
    lines = f.readlines()[-5:]
    for L in lines:
        print(L.strip())

---
### Notes
- The notebook includes simulations for streaming/IoT where setting up Kafka or a serial device is out of scope for a simple demo. These simulations are acceptable to show the concept in class.
- The Selenium cell is guarded because many environments don't have a webdriver installed; run it locally with chromedriver if you want real dynamic scraping.
- Files created by the notebook (CSV, Excel, DB, images, WAV, log) live in `/mnt/data` and are safe to push to GitHub if they are small. Remove generated binary files before pushing if you want a code-only repo.
